# Rule-Based Hierarchical System (RSHS) Classifier for LNUQNet

This notebook provides the initial lithology discrimation, rule-based lithology classification framework designed for generating ground truth data in the context of LNUQNet. It classifies well log data into three distinct lithology classes: Shale, Shaly Sandstone, and Sandstone. The classification relies on standard well log measurements such as Gamma Ray, Resistivity, Porosity, and Permeability.

The core functionality is encapsulated within the `WellLogClassifier` class, which offers methods for data pre-segmentation, facies-specific threshold calculation, and multi-stage lithology classification.

In [ ]:
from pathlib import Path
input_dir = Path("/content/input")
input_dir.mkdir(parents=True, exist_ok=True)
print("Exists:", input_dir.exists())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging

class WellLogClassifier:
    """
    Rule based lithology classification framework for robust
    ground truth generation in LNUQNet using three lithology classes.

    This classifier processes well log data to assign lithology labels
    (Shale, Shaly Sandstone, Sandstone) based on gamma ray, resistivity,
    porosity, and permeability measurements.
    """

    def __init__(self, facies_window=50):
        """
        Initializes the WellLogClassifier.

        Args:
            facies_window (int): Window size for rolling median calculation
                                 to determine facies segments.
        """
        self.facies_window = facies_window
        self.thresholds = None # Stores facies-specific thresholds for classification

        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

    # ------------------------------------------------------------------
    # Facies Pre-Segmentation
    # ------------------------------------------------------------------
    def compute_facies_segments(self, data):
        """
        Computes facies segments (Shale_Prone/Reservoir_Prone) based on
        a rolling median of Gamma Ray (GR) values.

        Args:
            data (pd.DataFrame): Input well log data with a 'Gamma.gAPI' column.

        Returns:
            pd.DataFrame: Data with an added 'GR_roll' and 'Facies' column.
        """
        data = data.copy()
        # Calculate rolling median of Gamma Ray to smooth out noise and identify trends
        data['GR_roll'] = data['Gamma.gAPI'].rolling(
            self.facies_window, center=True, min_periods=1
        ).median()

        # Determine a global median for GR to define initial facies zones
        gr_median = data['Gamma.gAPI'].median()
        # Assign 'Shale_Prone' if GR_roll is above global median, else 'Reservoir_Prone'
        data['Facies'] = np.where(
            data['GR_roll'] > gr_median,
            'Shale_Prone',
            'Reservoir_Prone'
        )
        return data

    # ------------------------------------------------------------------
    # Facies Specific Thresholds Calculation
    # ------------------------------------------------------------------
    def calculate_facies_thresholds(self, data):
        """
        Calculates facies-specific thresholds (GCP, RCP) for Gamma Ray and Resistivity.
        These thresholds are used in subsequent classification steps.

        Args:
            data (pd.DataFrame): Data with 'Facies', 'Gamma.gAPI', and 'Resistivity.ohm.m' columns.
        """
        thresholds = {}
        # Iterate through each facies segment to calculate its specific thresholds
        for facies, df in data.groupby('Facies'):
            thresholds[facies] = {
                'GCP': df['Gamma.gAPI'].quantile(0.75), # Gamma Ray Classification Point (75th percentile)
                'RCP': df['Resistivity.ohm.m'].quantile(0.25) # Resistivity Classification Point (25th percentile)
            }
        self.thresholds = thresholds

    # ------------------------------------------------------------------
    # Shale Classification
    # ------------------------------------------------------------------
    def classify_shale(self, row):
        """
        Classifies a row as Shale (0) based on facies-specific Gamma Ray
        and Resistivity thresholds.

        Args:
            row (pd.Series): A single row of well log data.

        Returns:
            int or None: 0 if classified as Shale, None otherwise.
        """
        # Retrieve facies-specific thresholds for the current row
        th = self.thresholds[row['Facies']]
        # Conditions for Shale: High Gamma Ray and Low Resistivity
        if (
            row['Gamma.gAPI'] > th['GCP'] and
            row['Resistivity.ohm.m'] < th['RCP']
        ):
            return 0 # Assign 0 for Shale
        return None

    # ------------------------------------------------------------------
    # Shaly Sandstone Classification
    # ------------------------------------------------------------------
    def classify_shaly_sandstone(self, row):
        """
        Classifies a row as Shaly Sandstone (1) based on transition Gamma Ray
        and other properties, excluding clean reservoir characteristics.

        Args:
            row (pd.Series): A single row of well log data.

        Returns:
            int or None: 1 if classified as Shaly Sandstone, None otherwise.
        """
        # Retrieve facies-specific thresholds
        th = self.thresholds[row['Facies']]

        # Check for Gamma Ray values in a transition zone (between clean and shaly)
        is_transition_gr = (
            row['Gamma.gAPI'] >= 0.85 * th['GCP'] and
            row['Gamma.gAPI'] <= th['GCP']
        )

        # Check if the rock is not 'clean' (i.e., not high porosity and high resistivity)
        is_not_clean = not (
            row['Porosity.m3/m3'] > 0.12 and # High porosity indicates clean sand
            row['Resistivity.ohm.m'] > th['RCP'] # High resistivity indicates clean sand
        )

        # If both transition GR and not clean, classify as Shaly Sandstone
        if is_transition_gr and is_not_clean:
            return 1 # Assign 1 for Shaly Sandstone
        return None

    # ------------------------------------------------------------------
    # Sandstone Candidacy Filter
    # ------------------------------------------------------------------
    def sandstone_candidate_filter(self, data):
        """
        Filters data to identify potential Sandstone candidates based on
        porosity, resistivity, and permeability criteria.

        Args:
            data (pd.DataFrame): Input well log data.

        Returns:
            pd.DataFrame: Filtered DataFrame containing only sandstone candidates.
        """
        return data[
            (
                (data['Porosity.m3/m3'] > 0.06) |
                (data['Resistivity.ohm.m'] > data['Resistivity.ohm.m'].quantile(0.5)) |
                (data['Perm.mD'] > 0.01)
            )
        ]

    # ------------------------------------------------------------------
    # Main Classification Pipeline
    # ------------------------------------------------------------------
    def classify_data(self, data):
        """
        Applies the three-class lithology classification pipeline to the well log data.
        The classification proceeds in stages: Shale, Shaly Sandstone, then Sandstone.
        Any unclassified samples are defaulted to Shale.

        Args:
            data (pd.DataFrame): Input well log data.

        Returns:
            pd.DataFrame: Data with an added 'Lithology' column containing classified labels (0, 1, or 2).
        """
        self.logger.info("Starting three class lithology classification")

        # Step 1: Pre-segment data into facies based on Gamma Ray
        data = self.compute_facies_segments(data)
        # Step 2: Calculate dynamic thresholds for each facies segment
        self.calculate_facies_thresholds(data)

        classified = data.copy()
        classified['Lithology'] = None # Initialize Lithology column with None

        # Stage 1: Classify Shale
        # Apply the Shale classification rule to all rows
        classified['Lithology'] = classified.apply(self.classify_shale, axis=1)

        # Stage 2: Classify Shaly Sandstone (only for unclassified rows)
        mask = classified['Lithology'].isna() # Identify rows not yet classified
        classified.loc[mask, 'Lithology'] = classified[mask].apply(
            self.classify_shaly_sandstone, axis=1
        )

        # Stage 3: Classify Sandstone (only for remaining unclassified rows)
        mask = classified['Lithology'].isna() # Identify rows still unclassified
        # Filter these unclassified rows for sandstone characteristics
        sandstone_candidates = self.sandstone_candidate_filter(classified[mask])
        # Assign 2 for Sandstone to the identified candidates
        classified.loc[sandstone_candidates.index, 'Lithology'] = 2

        # Fill any remaining unclassified samples as Shale (default to 0)
        classified['Lithology'] = classified['Lithology'].fillna(0).astype(int)

        self.logger.info("Classification completed")
        return classified

    # ------------------------------------------------------------------
    # Visualization
    # ------------------------------------------------------------------
    def plot_classifications(self, data, output_dir, well_name):
        """
        Generates and saves two plots visualizing the classification results:
        1. Porosity vs Permeability colored by lithology.
        2. Depth vs Lithology.

        Args:
            data (pd.DataFrame): Classified well log data with 'Lithology' column.
            output_dir (Path): Directory to save the plots.
            well_name (str): Name of the well for plot titles and filenames.
        """
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True) # Ensure output directory exists

        # Define lithology labels and corresponding colors for consistent plotting
        lithology_labels = ['Shale', 'Shaly Sandstone', 'Sandstone']
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green

        # Plot 1: Porosity vs Permeability
        plt.figure(figsize=(12, 8))
        for i in range(3):
            mask = data['Lithology'] == i
            if mask.any(): # Only plot if there are samples for this lithology
                plt.scatter(
                    data.loc[mask, 'Porosity.m3/m3'],
                    data.loc[mask, 'Perm.mD'],
                    label=lithology_labels[i],
                    color=colors[i],
                    alpha=0.6
                )

        plt.yscale('log') # Use logarithmic scale for permeability for better visualization
        plt.xlabel('Porosity (m3/m3)')
        plt.ylabel('Permeability (mD)')
        plt.title(f'Porosity vs Permeability: {well_name}')
        plt.legend()
        plt.grid(True, which='both', alpha=0.2)
        plt.savefig(output_dir / f'{well_name}_porosity_permeability.png', dpi=300)
        plt.close() # Close plot to free up memory

        # Plot 2: Depth vs Lithology
        plt.figure(figsize=(10, 20))
        plt.scatter(
            data['Lithology'],
            data['Depth.m'],
            c=data['Lithology'], # Color points by lithology
            cmap=plt.cm.colors.ListedColormap(colors),
            alpha=0.6
        )

        plt.gca().invert_yaxis() # Invert y-axis to show shallower depths at the top
        plt.xticks(range(3), lithology_labels, rotation=45) # Set custom x-axis ticks and labels
        plt.xlabel('Lithology')
        plt.ylabel('Depth (m)')
        plt.title(f'Depth vs Lithology: {well_name}')
        plt.grid(True, axis='y', alpha=0.2)
        plt.savefig(output_dir / f'{well_name}_depth_lithology.png', dpi=300)
        plt.close() # Close plot to free up memory

    # ------------------------------------------------------------------
    # Summary Statistics
    # ------------------------------------------------------------------
    def generate_summary(self, data, well_name):
        """
        Generates a summary of lithology counts for a given well.

        Args:
            data (pd.DataFrame): Classified well log data with 'Lithology' column.
            well_name (str): Name of the well.

        Returns:
            dict: A dictionary containing the well name and counts for each lithology type.
        """
        summary = data['Lithology'].value_counts().sort_index() # Count occurrences of each lithology
        return {
            'Well': well_name,
            'Shale': summary.get(0, 0),
            'Shaly Sandstone': summary.get(1, 0),
            'Sandstone': summary.get(2, 0)
        }


# ----------------------------------------------------------------------
# Main Execution Block
# ----------------------------------------------------------------------

def main():
    """
    Main function to orchestrate the lithology classification process.
    It reads well log data from input CSV files, classifies them,
    generates plots, and compiles a summary of classifications.
    """
    # Define input and output directories
    input_dir = Path("/content/input")
    output_dir = Path("/content/output")
    output_dir.mkdir(parents=True, exist_ok=True) # Create output directory if it doesn't exist
    input_dir.mkdir(parents=True, exist_ok=True) # Ensure input directory exists for file operations

    classifier = WellLogClassifier() # Instantiate the classifier
    summaries = [] # List to store summary statistics for each well

    # Process each CSV file (representing a well) in the input directory
    # Before processing, check if the input directory is empty and provide guidance.
    if not any(input_dir.iterdir()):
        logging.info(f"Input directory {input_dir} is empty. Please upload CSV files to this directory.")
        return # Exit if no files are found

    for csv_file in input_dir.glob('*.csv'):
        well_name = csv_file.stem # Extract well name from filename
        logging.info(f'Processing well {well_name}')

        data = pd.read_csv(csv_file) # Read well log data into a DataFrame
        classified = classifier.classify_data(data) # Perform lithology classification

        classifier.plot_classifications(classified, output_dir, well_name) # Generate and save plots
        summaries.append(classifier.generate_summary(classified, well_name)) # Generate and store summary

        # Save the classified data to a new CSV file in the output directory
        classified.to_csv(output_dir / f'classified_{csv_file.name}', index=False)

    # Save all well summaries into a single CSV file
    pd.DataFrame(summaries).to_csv(output_dir / 'well_summaries.csv', index=False)


if __name__ == '__main__':
    # Ensure main() is called only when the script is executed directly
    main()


## How to Use This Notebook and Expected Outputs

This notebook is designed to be run sequentially. Follow the steps below to successfully execute the lithology classification:

1.  **Run the initial setup cells**: Execute the code cells that define the `WellLogClassifier` class and the `main` function.
2.  **Prepare your input data**: Place your well log data as `.csv` files into the `/content/input` directory. Each `.csv` file should represent a single well and contain the following columns: 'Depth.m', 'Gamma.gAPI', 'Resistivity.ohm.m', 'Porosity.m3/m3', and 'Perm.mD'.
3.  **Execute the main classification block**: Run the code cell containing `if __name__ == '__main__': main()`. This will trigger the entire classification pipeline.

### Expected Outputs:

Upon successful execution, the following outputs will be generated in the `/content/output` directory:

*   **Classified CSV files**: For each input well (`well_name.csv`), a new file named `classified_well_name.csv` will be created. This file will include all original log curves plus a new 'Lithology' column, where:
    *   `0` represents Shale
    *   `1` represents Shaly Sandstone
    *   `2` represents Sandstone

*   **Visualization plots**: Two plots will be generated for each well:
    *   `well_name_porosity_permeability.png`: A scatter plot showing porosity vs. permeability, colored by the classified lithology, providing insight into reservoir quality.
    *   `well_name_depth_lithology.png`: A depth plot showing the lithology distribution along the wellbore.

*   **Summary CSV file**: A single `well_summaries.csv` file will be created, containing a table that summarizes the count of each lithology type for all processed wells.